# SmartAgro Score — Создание датасета (19 фич) + Обучение модели

Этот ноутбук объединяет два этапа:
1. **Feature Engineering** — загрузка `data.csv` → очистка → генерация 19 фич → `data_features.csv`
2. **Model Training** — XGBoost на 19 фичах (17 базовых + 2 норматива) → сохранение артефактов

### 19 фич:
- **17 базовых**: `gross_output_growth_yoy`, `land_to_livestock_ratio`, `historical_survival_rate`, `subsidy_dependence_index`, `veterinary_compliance`, `years_in_operation`, `pedigree_ratio`, `previous_subsidies_count`, `debt_load_ratio`, `log_amount`, `livestock_count`, `direction_code`, `is_pedigree`, `is_producer`, `hour_submitted`, `month_submitted`, `region_encoded`
- **+2 норматива**: `grazing_norm_deviation`, `natural_loss_risk_score`

In [ ]:
import json
import random
import warnings
from pathlib import Path
from datetime import datetime

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBRegressor

matplotlib.use("Agg")
warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# ── Пути ──
ROOT_DIR    = Path("..").resolve()
DATA_DIR    = ROOT_DIR / "data"
MODELS_DIR  = ROOT_DIR / "models"
REPORTS_DIR = ROOT_DIR / "reports"

MODELS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

INPUT_RAW    = DATA_DIR / "data.csv"
OUTPUT_CLEAN = DATA_DIR / "data_cleaned.csv"
OUTPUT_FEAT  = DATA_DIR / "data_features.csv"

print(f"📁 Root:    {ROOT_DIR}")
print(f"📁 Data:    {DATA_DIR}")
print(f"📁 Models:  {MODELS_DIR}")
print(f"📁 Reports: {REPORTS_DIR}")

## Часть 1: Feature Engineering

### 1.1 Загрузка и очистка сырых данных

In [ ]:
print(f"[1/5] Загружаю файл: {INPUT_RAW}")

df = pd.read_csv(INPUT_RAW, skiprows=3, header=0, low_memory=False)

df.columns = [
    "num", "date_received", "col3", "col4", "region", "agimat",
    "app_number", "direction", "subsidy_name", "status",
    "normative", "amount", "district",
]

df = df[df["num"] != "№ п/п"].copy()
df.dropna(how="all", inplace=True)

for col in ["region", "direction", "status", "subsidy_name", "district"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

df["normative"] = pd.to_numeric(df["normative"], errors="coerce")
df["amount"]    = pd.to_numeric(df["amount"],    errors="coerce")
df["num"]       = pd.to_numeric(df["num"],       errors="coerce")
df["date_received"] = pd.to_datetime(df["date_received"], errors="coerce", dayfirst=True)

df.drop(columns=["col3", "col4"], inplace=True, errors="ignore")
df = df[df["amount"].notna() & (df["amount"] > 0)].copy()

print(f"    ✅ Загружено строк: {len(df):,}")
print(f"    Колонки: {list(df.columns)}")

### 1.2 Базовые фичи из исходных данных

In [ ]:
print("\n[2/5] Создаю базовые фичи...")

df["hour_submitted"]   = df["date_received"].dt.hour.fillna(12)
df["day_of_week"]      = df["date_received"].dt.dayofweek.fillna(1)
df["month_submitted"]  = df["date_received"].dt.month.fillna(1)

df["livestock_count"] = (df["amount"] / df["normative"].replace(0, np.nan)).round(0)
df["livestock_count"] = df["livestock_count"].clip(lower=1).fillna(10)

df["log_amount"] = np.log1p(df["amount"])

# 9 направлений + fallback=9
DIRECTION_MAP = {
    "Субсидирование в скотоводстве":                          0,
    "Субсидирование в овцеводстве":                           1,
    "Субсидирование в коневодстве":                           2,
    "Субсидирование в птицеводстве":                          3,
    "Субсидирование в верблюдоводстве":                       4,
    "Субсидирование в свиноводстве":                          5,
    "Субсидирование в козоводстве":                           6,
    "Субсидирование в пчеловодстве":                          7,
    "Субсидирование затрат по искусственному осеменению":     8,
}
df["direction_code"] = df["direction"].map(DIRECTION_MAP).fillna(9)

df["is_pedigree"] = (
    df["subsidy_name"].str.contains("племен", case=False, na=False)
).astype(int)

df["is_producer"] = (
    df["subsidy_name"].str.contains("производи|производит", case=False, na=False)
).astype(int)

print(f"    ✅ Базовых фич добавлено: 8")

### 1.3 Синтетические экономические фичи + нормативы

In [ ]:
print("\n[3/5] Генерирую синтетические экономические фичи...")
n = len(df)

# ── gross_output_growth_yoy ──
base_growth = (df["log_amount"] - df["log_amount"].mean()) / df["log_amount"].std() * 0.1
pedigree_bonus = df["is_pedigree"] * np.random.uniform(0.05, 0.15, n)
noise_growth = np.random.normal(0, 0.12, n)
df["gross_output_growth_yoy"] = (base_growth + pedigree_bonus + noise_growth).clip(-0.30, 0.80)

# ── land_to_livestock_ratio ──
direction_land_factor = df["direction_code"].map({
    0: 3.0,   # скотоводство
    1: 2.5,   # овцеводство
    2: 2.0,   # коневодство
    3: 0.3,   # птицеводство
    4: 4.0,   # верблюдоводство
    5: 0.5,   # свиноводство
    6: 2.5,   # козоводство
    7: 0.2,   # пчеловодство
    8: 1.5,   # искусственное осеменение
    9: 2.0,   # неизвестное
}).fillna(2.0)
df["land_to_livestock_ratio"] = (
    direction_land_factor * np.random.lognormal(0, 0.4, n)
).clip(0.2, 10.0)

# ── historical_survival_rate ──
region_survival = {
    "Мангистауская область":       0.82,
    "Атырауская область":          0.83,
    "Западно-Казахстанская область": 0.85,
    "Жамбылская область":          0.87,
    "Алматинская область":         0.90,
    "Акмолинская область":         0.88,
    "область Ұлытау":              0.86,
    "область Жетісу":              0.89,
    "г.Шымкент":                   0.87,
}
base_survival = df["region"].map(region_survival).fillna(0.87)
noise_survival = np.random.normal(0, 0.05, n)
bird_bonus = (df["direction_code"] == 3) * 0.04
df["historical_survival_rate"] = (
    base_survival + noise_survival + bird_bonus
).clip(0.50, 0.99)

# ── subsidy_dependence_index ──
estimated_revenue = df["livestock_count"] * direction_land_factor * np.random.uniform(50000, 200000, n)
raw_dependence = df["amount"] / (estimated_revenue + df["amount"])
df["subsidy_dependence_index"] = raw_dependence.clip(0.0, 1.0)

# ── veterinary_compliance ──
noise_vet = np.random.beta(8, 2, n)
pedigree_compliance_bonus = df["is_pedigree"] * 0.05
df["veterinary_compliance"] = (noise_vet + pedigree_compliance_bonus).clip(0.0, 1.0)

# ── years_in_operation ──
probs = np.array([1 / (1 + y * 0.3) for y in range(25)])
probs /= probs.sum()
df["years_in_operation"] = np.random.choice(range(1, 26), n, p=probs).astype(float)

# ── pedigree_ratio ──
df["pedigree_ratio"] = np.where(
    df["is_pedigree"] == 1,
    np.random.beta(5, 2, n),
    np.random.beta(2, 5, n),
)

# ── previous_subsidies_count ──
df["previous_subsidies_count"] = np.random.poisson(lam=3.5, size=n).clip(0, 15)

# ── debt_load_ratio ──
df["debt_load_ratio"] = np.random.lognormal(mean=0.3, sigma=0.7, size=n).clip(0.0, 5.0)

print(f"    ✅ Синтетических фич добавлено: 9")

### 1.4 Нормативные фичи (grazing + natural loss)

In [ ]:
print("\n[3b/5] Генерирую нормативные фичи...")

# ── GRAZING_NORM_HA ──
GRAZING_NORM_HA = {
    ("Акмолинская область", 0): 8.25,    ("Акмолинская область", 1): 1.65,
    ("Акмолинская область", 2): 9.90,    ("Акмолинская область", 4): 11.55,
    ("Мангистауская область", 0): 13.00, ("Мангистауская область", 1): 2.60,
    ("Мангистауская область", 2): 15.60, ("Мангистауская область", 4): 20.40,
    ("Алматинская область", 0): 11.25,   ("Алматинская область", 1): 2.25,
    ("Алматинская область", 2): 13.50,   ("Алматинская область", 4): 13.70,
    ("Актюбинская область", 0): 11.75,   ("Актюбинская область", 1): 2.35,
    ("Актюбинская область", 2): 14.10,   ("Актюбинская область", 4): 16.45,
    ("Атырауская область", 0): 12.00,    ("Атырауская область", 1): 2.50,
    ("Атырауская область", 2): 14.40,    ("Атырауская область", 4): 18.00,
    ("Западно-Казахстанская область", 0): 7.50,
    ("Западно-Казахстанская область", 1): 1.50,
    ("Западно-Казахстанская область", 2): 9.00,
    ("Западно-Казахстанская область", 4): 10.50,
    ("Восточно-Казахстанская область", 0): 9.00,
    ("Восточно-Казахстанская область", 1): 1.80,
    ("Восточно-Казахстанская область", 2): 10.80,
    ("Восточно-Казахстанская область", 4): 12.60,
    ("Карагандинская область", 0): 10.00, ("Карагандинская область", 1): 2.00,
    ("Карагандинская область", 2): 12.00, ("Карагандинская область", 4): 14.00,
    ("Костанайская область", 0): 7.00,    ("Костанайская область", 1): 1.40,
    ("Костанайская область", 2): 8.40,    ("Костанайская область", 4): 9.80,
    ("Кызылординская область", 0): 11.00, ("Кызылординская область", 1): 2.20,
    ("Кызылординская область", 2): 13.20, ("Кызылординская область", 4): 15.40,
    ("Павлодарская область", 0): 8.50,    ("Павлодарская область", 1): 1.70,
    ("Павлодарская область", 2): 10.20,   ("Павлодарская область", 4): 11.90,
    ("Северо-Казахстанская область", 0): 6.50,
    ("Северо-Казахстанская область", 1): 1.30,
    ("Северо-Казахстанская область", 2): 7.80,
    ("Северо-Казахстанская область", 4): 9.10,
    ("Туркестанская область", 0): 9.50,   ("Туркестанская область", 1): 1.90,
    ("Туркестанская область", 2): 11.40,  ("Туркестанская область", 4): 13.30,
    ("Жамбылская область", 0): 10.50,     ("Жамбылская область", 1): 2.10,
    ("Жамбылская область", 2): 12.60,     ("Жамбылская область", 4): 14.70,
    ("область Абай", 0): 9.00,            ("область Абай", 1): 1.80,
    ("область Абай", 2): 10.80,           ("область Абай", 4): 12.60,
    ("область Ұлытау", 0): 10.00,         ("область Ұлытау", 1): 2.00,
    ("область Ұлытау", 2): 12.00,         ("область Ұлытау", 4): 14.00,
    ("область Жетісу", 0): 10.50,         ("область Жетісу", 1): 2.10,
    ("область Жетісу", 2): 12.60,         ("область Жетісу", 4): 14.70,
    ("г.Шымкент", 0): 9.50,               ("г.Шымкент", 1): 1.90,
    ("г.Шымкент", 2): 11.40,              ("г.Шымкент", 4): 13.30,
}
GRAZING_NORM_DEFAULT = 5.0

grazing_norm = df.apply(
    lambda r: GRAZING_NORM_HA.get(
        (r["region"], int(r["direction_code"])),
        GRAZING_NORM_DEFAULT
    ),
    axis=1
)
grazing_actual = grazing_norm * np.random.lognormal(0, 0.3, n)
df["grazing_norm_deviation"] = (
    (grazing_actual - grazing_norm) / grazing_norm
).clip(-2.0, 2.0)

# ── MORTALITY_NORM ──
MORTALITY_NORM = {
    0: 0.025,   # КРС
    1: 0.030,   # Овцы
    2: 0.027,   # Лошади
    3: 0.075,   # Птица
    4: 0.022,   # Верблюды
    5: 0.010,   # Свиньи
    6: 0.028,   # Козы
    7: 0.050,   # Пчёлы
    8: 0.025,   # Искусственное осеменение
    9: 0.030,   # Неизвестное
}

mortality_norm = df["direction_code"].map(MORTALITY_NORM).fillna(0.03)
actual_mortality = mortality_norm * np.random.lognormal(0, 0.5, n)
df["natural_loss_risk_score"] = (
    actual_mortality / mortality_norm
).clip(0.0, 3.0)

print(f"    ✅ Нормативных фич добавлено: 2")

### 1.5 Целевая переменная (historical_score)

In [ ]:
print("\n[4/5] Создаю целевую переменную (historical_score)...")

def norm(series):
    rng = series.max() - series.min()
    return (series - series.min()) / rng if rng > 0 else series * 0

debt_inverted     = 1 - norm(df["debt_load_ratio"])
dependence_inv    = 1 - norm(df["subsidy_dependence_index"])
grazing_inverted  = norm(df["grazing_norm_deviation"] + 2.0)
risk_inverted     = 1 - norm(df["natural_loss_risk_score"])

raw_score = (
    norm(df["gross_output_growth_yoy"])    * 22.0 +
    norm(df["pedigree_ratio"])              * 18.0 +
    norm(df["historical_survival_rate"])    * 13.0 +
    norm(df["veterinary_compliance"])       * 11.0 +
    norm(df["subsidy_dependence_index"].apply(lambda x: 1-x))  * 10.0 +
    debt_inverted                           *  9.0 +
    norm(df["land_to_livestock_ratio"])     *  4.0 +
    norm(df["years_in_operation"])          *  4.0 +
    grazing_inverted                        *  5.0 +
    risk_inverted                           *  4.0
)

raw_norm = (raw_score - raw_score.min()) / (raw_score.max() - raw_score.min())
df["historical_score"] = (raw_norm * 99 + 1).round(1)

noise = np.random.normal(0, 3, len(df))
df["historical_score"] = (df["historical_score"] + noise).clip(1, 100).round(1)

print(f"    Score — среднее: {df['historical_score'].mean():.1f}")
print(f"    Score — мин: {df['historical_score'].min():.1f}, макс: {df['historical_score'].max():.1f}")
print(f"    🟢 Зелёных (80+): {(df['historical_score'] >= 80).sum():,}")
print(f"    🟡 Жёлтых (50-79): {((df['historical_score'] >= 50) & (df['historical_score'] < 80)).sum():,}")
print(f"    🔴 Красных (<50): {(df['historical_score'] < 50).sum():,}")

### 1.6 Кодирование категориальных переменных + сохранение

In [ ]:
print("\n[5/5] Кодирую категориальные переменные...")

le_region = LabelEncoder()
le_direction = LabelEncoder()

df["region_encoded"]    = le_region.fit_transform(df["region"].fillna("Неизвестно"))
df["direction_encoded"] = le_direction.fit_transform(df["direction"].fillna("Неизвестно"))

region_mapping = dict(zip(le_region.classes_, le_region.transform(le_region.classes_)))
print(f"    ✅ Регионов закодировано: {len(region_mapping)}")
print(f"    ✅ Направлений закодировано: {le_direction.classes_.tolist()}")

# ── Сохранение ──
df.to_csv(OUTPUT_CLEAN, index=False, encoding="utf-8-sig")
print(f"\n✅ Сохранён: {OUTPUT_CLEAN} ({len(df):,} строк, {df.shape[1]} колонок)")

# ── ML-фичи (19 шт) ──
ML_FEATURES = [
    "gross_output_growth_yoy", "land_to_livestock_ratio",
    "historical_survival_rate", "subsidy_dependence_index",
    "veterinary_compliance", "years_in_operation",
    "pedigree_ratio", "previous_subsidies_count", "debt_load_ratio",
    "grazing_norm_deviation", "natural_loss_risk_score",
    "log_amount", "livestock_count", "direction_code",
    "is_pedigree", "is_producer", "hour_submitted",
    "month_submitted", "region_encoded",
]
TARGET = "historical_score"

df_ml = df[ML_FEATURES + [TARGET]].dropna()
df_ml.to_csv(OUTPUT_FEAT, index=False, encoding="utf-8-sig")
print(f"✅ Сохранён: {OUTPUT_FEAT} ({len(df_ml):,} строк, {len(ML_FEATURES)} фичей)")

print("\n📊 Статистика датасета для ML:")
display(df_ml[ML_FEATURES].describe().round(3))

print("\n🎯 Распределение целевой переменной:")
print(f"   🟢 Зелёная зона (80-100): {(df_ml[TARGET] >= 80).mean()*100:.1f}%")
print(f"   🟡 Жёлтая зона  (50-79):  {((df_ml[TARGET] >= 50) & (df_ml[TARGET] < 80)).mean()*100:.1f}%")
print(f"   🔴 Красная зона (1-49):   {(df_ml[TARGET] < 50).mean()*100:.1f}%")

## Часть 2: Обучение модели (XGBoost, 19 фич)

### 2.1 Загрузка и split данных

In [ ]:
print(f"[1/6] Загружаю датасет: {OUTPUT_FEAT}")

df_feat = pd.read_csv(OUTPUT_FEAT)
print(f"    Загружено: {len(df_feat):,} строк, {df_feat.shape[1]} колонок")

missing = [f for f in ML_FEATURES if f not in df_feat.columns]
if missing:
    raise ValueError(f"❌ Отсутствуют фичи: {missing}")

X = df_feat[ML_FEATURES].copy()
y = df_feat[TARGET].copy()

mask = X.notna().all(axis=1) & y.notna()
X, y = X[mask], y[mask]
print(f"    После очистки: {len(X):,} строк")
print(f"    Фичей: {len(ML_FEATURES)}")
print(f"    Таргет — мин: {y.min():.1f}, макс: {y.max():.1f}, среднее: {y.mean():.1f}")

# ── Split 80/20 ──
print("\n[2/6] Делю данные на train/test (80/20)...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED,
)
print(f"    Train: {len(X_train):,} строк ({len(X_train)/len(X)*100:.0f}%)")
print(f"    Test:  {len(X_test):,}  строк ({len(X_test)/len(X)*100:.0f}%)")

### 2.2 Нормализация (StandardScaler)

In [ ]:
print("\n[3/6] Нормализую данные (StandardScaler)...")

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=ML_FEATURES, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=ML_FEATURES, index=X_test.index
)

print(f"    Среднее после скейлинга (expect ~0): {X_train_scaled.mean().mean():.4f}")
print(f"    Std после скейлинга (expect ~1):     {X_train_scaled.std().mean():.4f}")

### 2.3 Обучение XGBoost

In [ ]:
print("\n[4/6] Обучаю XGBoost модель...")
print("    Это может занять 30-60 секунд...")

model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_lambda=1.0,
    reg_alpha=0.1,
    min_child_weight=5,
    early_stopping_rounds=50,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbosity=0,
)

model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_test_scaled, y_test)],
    verbose=50,
)

print(f"\n    ✅ Обучено деревьев: {model.best_iteration + 1} (из 500)")
print(f"    Лучший тест MAE: {model.best_score:.3f}")

### 2.4 Оценка качества модели

In [ ]:
print("\n[5/6] Оцениваю качество модели...")

y_pred_test  = np.clip(model.predict(X_test_scaled),  1, 100)
y_pred_train = np.clip(model.predict(X_train_scaled), 1, 100)

metrics = {
    "train": {
        "MAE":  round(mean_absolute_error(y_train, y_pred_train), 3),
        "RMSE": round(np.sqrt(mean_squared_error(y_train, y_pred_train)), 3),
        "R2":   round(r2_score(y_train, y_pred_train), 4),
    },
    "test": {
        "MAE":  round(mean_absolute_error(y_test, y_pred_test), 3),
        "RMSE": round(np.sqrt(mean_squared_error(y_test, y_pred_test)), 3),
        "R2":   round(r2_score(y_test, y_pred_test), 4),
    },
}

# 5-fold CV
print("    Запускаю 5-fold Cross-Validation...")
cv_model = XGBRegressor(
    n_estimators=200, max_depth=6, learning_rate=0.05,
    random_state=RANDOM_SEED, n_jobs=-1, verbosity=0,
)
cv_scores = cross_val_score(
    cv_model, X_train_scaled, y_train,
    cv=5, scoring="neg_mean_absolute_error", n_jobs=-1,
)
cv_mae = -cv_scores.mean()
metrics["cross_val_mae"] = round(cv_mae, 3)
metrics["cv_std"] = round(cv_scores.std(), 3)

# MAE по зонам
df_errors = pd.DataFrame({"y_true": y_test, "y_pred": y_pred_test})
df_errors["zone"] = pd.cut(
    df_errors["y_true"], bins=[0, 50, 80, 100], labels=["red", "yellow", "green"]
)
zone_errors = df_errors.groupby("zone", observed=True).apply(
    lambda g: round(mean_absolute_error(g["y_true"], g["y_pred"]), 2)
).to_dict()
metrics["mae_by_zone"] = zone_errors

# ── Вывод таблицы ──
print("\n    ┌─────────────────────────────────────────┐")
print("    │           МЕТРИКИ КАЧЕСТВА МОДЕЛИ       │")
print("    ├────────────────┬────────────┬───────────┤")
print("    │ Метрика        │ Train      │ Test      │")
print("    ├────────────────┼────────────┼───────────┤")
print(f"    │ MAE (баллы)    │ {metrics['train']['MAE']:<10} │ {metrics['test']['MAE']:<9} │")
print(f"    │ RMSE (баллы)   │ {metrics['train']['RMSE']:<10} │ {metrics['test']['RMSE']:<9} │")
print(f"    │ R² (0-1)       │ {metrics['train']['R2']:<10} │ {metrics['test']['R2']:<9} │")
print("    ├────────────────┴────────────┴───────────┤")
print(f"    │ Cross-Val MAE (5-fold): {cv_mae:.3f} ± {metrics['cv_std']:.3f}     │")
print("    ├─────────────────────────────────────────┤")
print("    │ MAE по зонам:                           │")
for zone, mae in zone_errors.items():
    print(f"    │   {zone:<10}: {mae} баллов{' ' * (20 - len(str(mae)))}│")
print("    └─────────────────────────────────────────┘")

overfit_gap = abs(metrics["train"]["MAE"] - metrics["test"]["MAE"])
if overfit_gap < 1.0:
    print("\n    ✅ Переобучения НЕТ (Train MAE ≈ Test MAE)")
elif overfit_gap < 3.0:
    print(f"\n    ⚠️  Небольшое переобучение (разрыв: {overfit_gap:.1f} балла)")
else:
    print(f"\n    ❌ Переобучение! Разрыв Train/Test MAE: {overfit_gap:.1f} балла")

### 2.5 Feature Importance

In [ ]:
print("\n    Строю график важности фичей...")

importances = pd.Series(model.feature_importances_, index=ML_FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ["#d32f2f" if imp < importances.median() else "#1976d2" for imp in importances]
importances.plot(kind="barh", ax=ax, color=colors)
ax.set_title("Важность фичей XGBoost (19 фич)", fontsize=14, fontweight="bold")
ax.set_xlabel("Важность (чем больше, тем важнее фича)")
ax.axvline(importances.median(), color="orange", linestyle="--", alpha=0.7, label="Медиана")
ax.legend()
plt.tight_layout()
plt.savefig(REPORTS_DIR / "feature_importance_19.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"    ✅ Сохранён: {REPORTS_DIR / 'feature_importance_19.png'}")

# ── Топ-10 фич ──
print("\n📊 Топ-10 самых важных фич:")
top10 = importances.sort_values(ascending=False).head(10)
for i, (feat, imp) in enumerate(top10.items(), 1):
    print(f"  {i:2d}. {feat:<35s} {imp:.4f}")

### 2.6 SHAP Explainer

In [ ]:
print("\n    Создаю SHAP TreeExplainer...")
explainer = shap.TreeExplainer(model)
print("    ✅ SHAP Explainer готов!")

# ── SHAP summary plot ──
shap_values = explainer.shap_values(X_train_scaled)
fig, ax = plt.subplots(figsize=(10, 7))
shap.summary_plot(shap_values, X_train_scaled, show=False, plot_size=(10, 7))
plt.tight_layout()
plt.savefig(REPORTS_DIR / "shap_summary_19.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"    ✅ Сохранён: {REPORTS_DIR / 'shap_summary_19.png'}")

### 2.7 Сохранение артефактов модели

In [ ]:
print("\n[6/6] Сохраняю артефакты модели...")

joblib.dump(model,     MODELS_DIR / "xgb_scorer.joblib")
joblib.dump(scaler,    MODELS_DIR / "scaler.joblib")
joblib.dump(explainer, MODELS_DIR / "shap_explainer.joblib")

with open(MODELS_DIR / "feature_names.json", "w", encoding="utf-8") as f:
    json.dump(ML_FEATURES, f, ensure_ascii=False, indent=2)

# ── Отчёт ──
report_text = f"""SmartAgro Score — Model Training Report
========================================
Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Model: XGBoost Regressor
Features: {len(ML_FEATURES)} (19 фич: 17 базовых + 2 норматива)

METRICS:
  Train MAE:  {metrics['train']['MAE']} баллов
  Train RMSE: {metrics['train']['RMSE']} баллов
  Train R²:   {metrics['train']['R2']}

  Test MAE:   {metrics['test']['MAE']} баллов
  Test RMSE:  {metrics['test']['RMSE']} баллов
  Test R²:    {metrics['test']['R2']}

  Cross-Val MAE (5-fold): {metrics['cross_val_mae']} ± {metrics['cv_std']}

  MAE по зонам: {metrics['mae_by_zone']}

FEATURE LIST:
{''.join(f'  {i+1}. {f}\n' for i, f in enumerate(ML_FEATURES))}
"""
with open(REPORTS_DIR / "model_report_19.txt", "w", encoding="utf-8") as f:
    f.write(report_text)

print(f"    ✅ {MODELS_DIR / 'xgb_scorer.joblib'}")
print(f"    ✅ {MODELS_DIR / 'scaler.joblib'}")
print(f"    ✅ {MODELS_DIR / 'shap_explainer.joblib'}")
print(f"    ✅ {MODELS_DIR / 'feature_names.json'}")
print(f"    ✅ {REPORTS_DIR / 'model_report_19.txt'}")
print(f"    ✅ {REPORTS_DIR / 'feature_importance_19.png'}")
print(f"    ✅ {REPORTS_DIR / 'shap_summary_19.png'}")

### 2.8 ДЕМО: Предсказание для одного фермера

In [ ]:
print("\n" + "=" * 55)
print("  ДЕМО: Предсказание для одного фермера")
print("=" * 55)

one_farmer = X_test.iloc[[0]]
true_score = y_test.iloc[0]

one_farmer_scaled = scaler.transform(one_farmer)
predicted_score = float(np.clip(model.predict(one_farmer_scaled)[0], 1, 100))

print(f"\n  Реальный балл:     {true_score:.1f}")
print(f"  Предсказанный:     {predicted_score:.1f}")
print(f"  Ошибка:            {abs(predicted_score - true_score):.1f} баллов")

if predicted_score >= 80:
    zone = "🟢 GREEN — Строго рекомендовано"
elif predicted_score >= 50:
    zone = "🟡 YELLOW — Требует рассмотрения"
else:
    zone = "🔴 RED — Не рекомендовано"
print(f"  Зона:              {zone}")

print("\n  Данные этого фермера:")
for feat, val in one_farmer.iloc[0].items():
    print(f"    {feat}: {val:.4f}")

### 2.9 Итоговый вывод

In [ ]:
print("=" * 65)
print("  ✨ ОБУЧЕНИЕ ЗАВЕРШЕНО!")
print("=" * 65)
print(f"\n  Модель: XGBoost, 19 фич")
print(f"  Train MAE:  {metrics['train']['MAE']:.3f}  |  R²: {metrics['train']['R2']:.4f}")
print(f"  Test  MAE:  {metrics['test']['MAE']:.3f}  |  R²: {metrics['test']['R2']:.4f}")
print(f"  CV MAE:     {metrics['cross_val_mae']:.3f} ± {metrics['cv_std']:.3f}")
print(f"  Деревьев:   {model.best_iteration + 1}")
print(f"\n  📁 Модель:     {MODELS_DIR / 'xgb_scorer.joblib'}")
print(f"  📁 Скейлер:    {MODELS_DIR / 'scaler.joblib'}")
print(f"  📁 SHAP:       {MODELS_DIR / 'shap_explainer.joblib'}")
print(f"  📁 Фичи:       {MODELS_DIR / 'feature_names.json'}")
print(f"  📁 Отчёт:      {REPORTS_DIR / 'model_report_19.txt'}")
print(f"\n  Следующий шаг: запустите фронтенд или API")
print("=" * 65)